# Chapter 3: Variations, Configuration Errors & Transport Maps

This Colab-ready notebook numerically verifies the core geometric identities from Chapter 3 of *A Lie-Interpretation of Robotics and Control*:
1. **Compatibility of Transport Maps**: $d_2 \Psi = -T^*(q, q_d)\, d_1 \Psi$ on $\mathbb{S}^2$ and $SO(3)$.
2. **Time-Derivative Identities on $SO(3)$**: $\dot{\Psi}_R = e_R \cdot e_\Omega$ and $\dot{e}_R = C(R_d^\top R)\, e_\Omega$.
3. **First-Order Equivalence**: Convergence of $(e_R^\epsilon/\epsilon,\, e_\Omega^\epsilon/\epsilon + \Omega_d^\wedge \eta) \to (\eta,\, \delta\Omega)$ at $\mathcal{O}(\epsilon)$ (residual $\mathcal{O}(\epsilon^2)$) as $\epsilon \to 0$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)
rng = np.random.default_rng(42)

def hat(x: np.ndarray) -> np.ndarray:
    """Hat map: R^3 -> so(3) with hat(x) @ y = np.cross(x, y)."""
    x = np.asarray(x, dtype=float)
    return np.array([
        [0.0, -x[2], x[1]],
        [x[2], 0.0, -x[0]],
        [-x[1], x[0], 0.0]
    ])

def vee(S: np.ndarray) -> np.ndarray:
    """Vee map: so(3) -> R^3."""
    return np.array([S[2, 1], S[0, 2], S[1, 0]], dtype=float)

def exp_so3(phi: np.ndarray) -> np.ndarray:
    """Rodrigues' exponential map exp(hat(phi)) on SO(3)."""
    theta = np.linalg.norm(phi)
    if theta < 1e-12:
        K = hat(phi)
        return np.eye(3) + K + 0.5 * (K @ K)
    K = hat(phi / theta)
    return np.eye(3) + np.sin(theta) * K + (1.0 - np.cos(theta)) * (K @ K)

def random_so3(rng) -> np.ndarray:
    return exp_so3(rng.normal(size=3))

def random_s2(rng) -> np.ndarray:
    v = rng.normal(size=3)
    return v / np.linalg.norm(v)

print('Lie algebra utilities initialized.')

## Test (i): Compatibility Condition $d_2\Psi = -T^*(q, q_d)\, d_1\Psi$ on $\mathbb{S}^2$ and $SO(3)$

We verify both:
- **$\mathbb{S}^2$**: $\Psi_q = 1 - q \cdot q_d$, $d_1\Psi_q = q^\wedge{}^2 q_d$, $d_2\Psi_q = (q_d^\wedge)^2 q = e_q$, $T(q, q_d)\dot{q}_d = (q_d \times \dot{q}_d) \times q$, and $T^*(q, q_d)v = (q \times v) \times q_d$.
- **$SO(3)$**: $\Psi_R = \frac{1}{2}\operatorname{Tr}(I_3 - R_d^\top R)$, $d_1\Psi_R = e_R = \frac{1}{2}(R_d^\top R - R^\top R_d)^\vee$, $d_2\Psi_R = -e_R$, $T(R, R_d)\Omega_d = R^\top R_d \Omega_d$, and $T^*(R, R_d) e_R = R_d^\top R\, e_R = e_R$.

In [ ]:
# --- S^2 Compatibility Verification ---
max_err_s2_fd = 0.0
max_err_s2_compat = 0.0
h = 1e-7

for _ in range(50):
    q = random_s2(rng)
    qd = random_s2(rng)
    psi_q = lambda a, b: 1.0 - np.dot(a, b)

    d1_psi = hat(q) @ hat(q) @ qd
    d2_psi = hat(qd) @ hat(qd) @ q  # e_q

    # Finite-difference check of d1_psi and d2_psi along tangent directions
    xi1 = rng.normal(size=3); xi1 -= np.dot(xi1, q) * q
    xi2 = rng.normal(size=3); xi2 -= np.dot(xi2, qd) * qd
    q_p = exp_so3(h * xi1) @ q; q_m = exp_so3(-h * xi1) @ q
    qd_p = exp_so3(h * xi2) @ qd; qd_m = exp_so3(-h * xi2) @ qd

    fd1 = (psi_q(q_p, qd) - psi_q(q_m, qd)) / (2 * h)
    fd2 = (psi_q(q, qd_p) - psi_q(q, qd_m)) / (2 * h)
    an1 = np.dot(d1_psi, np.cross(xi1, q))
    an2 = np.dot(d2_psi, np.cross(xi2, qd))
    max_err_s2_fd = max(max_err_s2_fd, abs(fd1 - an1), abs(fd2 - an2))

    # Adjoint T*(q, qd) v = (q x v) x qd
    T_star_d1 = np.cross(np.cross(q, d1_psi), qd)
    max_err_s2_compat = max(max_err_s2_compat, np.linalg.norm(d2_psi + T_star_d1))

# --- SO(3) Compatibility Verification ---
max_err_so3_fd = 0.0
max_err_so3_compat = 0.0

for _ in range(50):
    R = random_so3(rng)
    Rd = random_so3(rng)
    psi_R = lambda A, B: 0.5 * np.trace(np.eye(3) - B.T @ A)

    e_R = 0.5 * vee(Rd.T @ R - R.T @ Rd)
    d1_psi_R = e_R
    d2_psi_R = -e_R

    zeta = rng.normal(size=3)
    eta = rng.normal(size=3)
    fd1 = (psi_R(R @ exp_so3(h * zeta), Rd) - psi_R(R @ exp_so3(-h * zeta), Rd)) / (2 * h)
    fd2 = (psi_R(R, Rd @ exp_so3(h * eta)) - psi_R(R, Rd @ exp_so3(-h * eta))) / (2 * h)
    max_err_so3_fd = max(max_err_so3_fd, abs(fd1 - np.dot(d1_psi_R, zeta)), abs(fd2 - np.dot(d2_psi_R, eta)))

    # T(R, Rd) Omega_d = R^T Rd Omega_d => T*(R, Rd) e_R = Rd^T R e_R
    T_star_d1_R = Rd.T @ R @ d1_psi_R
    max_err_so3_compat = max(max_err_so3_compat, np.linalg.norm(d2_psi_R + T_star_d1_R))

print(f'S^2  FD differential error:  {max_err_s2_fd:.3e}')
print(f'S^2  d2Psi + T* d1Psi error: {max_err_s2_compat:.3e}')
print(f'SO(3) FD differential error: {max_err_so3_fd:.3e}')
print(f'SO(3) d2Psi + T* d1Psi err:  {max_err_so3_compat:.3e}')
assert max_err_s2_fd < 1e-8 and max_err_s2_compat < 1e-14
assert max_err_so3_fd < 1e-8 and max_err_so3_compat < 1e-14

## Test (ii): Time-Derivative Identities $\dot{\Psi}_R = e_R \cdot e_\Omega$ and $\dot{e}_R = C(R_d^\top R)\, e_\Omega$

We integrate smooth time-varying trajectories $R(t), R_d(t) \in SO(3)$ driven by $\Omega(t), \Omega_d(t)$ and compare central finite-difference time derivatives of $\Psi_R(t)$ and $e_R(t)$ against the exact analytic formulas.

In [ ]:
dt = 1e-4
T_end = 4.0
ts = np.arange(0.0, T_end + dt, dt)
N = len(ts)

R = exp_so3(np.array([0.4, -0.3, 0.5]))
Rd = np.eye(3)

Psi_hist = np.zeros(N)
eR_hist = np.zeros((N, 3))
eOm_hist = np.zeros((N, 3))
C_eOm_hist = np.zeros((N, 3))

def omega_fn(t):
    return np.array([0.7 * np.cos(1.3 * t), -0.5 * np.sin(0.9 * t), 0.4 * np.cos(0.5 * t)])

def omega_d_fn(t):
    return np.array([0.3 * np.sin(1.1 * t), 0.6 * np.cos(0.7 * t), -0.2 * np.sin(1.5 * t)])

for k, t in enumerate(ts):
    Om = omega_fn(t)
    Omd = omega_d_fn(t)
    A = Rd.T @ R
    Psi_hist[k] = 0.5 * np.trace(np.eye(3) - A)
    eR = 0.5 * vee(A - A.T)
    eOm = Om - R.T @ Rd @ Omd
    C_A = 0.5 * (np.trace(A) * np.eye(3) - A.T)

    eR_hist[k] = eR
    eOm_hist[k] = eOm
    C_eOm_hist[k] = C_A @ eOm

    # Midpoint Lie-group integration
    if k < N - 1:
        R = R @ exp_so3(omega_fn(t + 0.5 * dt) * dt)
        Rd = Rd @ exp_so3(omega_d_fn(t + 0.5 * dt) * dt)

Psi_dot_num = (Psi_hist[2:] - Psi_hist[:-2]) / (2 * dt)
Psi_dot_ana = np.sum(eR_hist[1:-1] * eOm_hist[1:-1], axis=1)
err_Psi_dot = np.abs(Psi_dot_num - Psi_dot_ana)

eR_dot_num = (eR_hist[2:] - eR_hist[:-2]) / (2 * dt)
eR_dot_ana = C_eOm_hist[1:-1]
err_eR_dot = np.linalg.norm(eR_dot_num - eR_dot_ana, axis=1)

print(f'Max |Psi_dot_num - eR . eOm|:        {err_Psi_dot.max():.3e}')
print(f'Max ||eR_dot_num - C(Rd^T R) eOm||:  {err_eR_dot.max():.3e}')
assert err_Psi_dot.max() < 1e-7 and err_eR_dot.max() < 1e-7

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(ts[1:-1], err_Psi_dot + 1e-16, label=r'$|\dot{\Psi}_R - e_R \cdot e_\Omega|$')
ax.semilogy(ts[1:-1], err_eR_dot + 1e-16, label=r'$\|\dot{e}_R - C(R_d^\top R)e_\Omega\|$')
ax.set_xlabel('Time $t$ [s]')
ax.set_ylabel('Residual')
ax.set_title('SO(3) Time-Derivative Identity Verification')
ax.grid(True, which='both', ls='--', alpha=0.4)
ax.legend()
plt.tight_layout()
plt.show()

## Test (iii): First-Order Convergence $(\eta, \delta\Omega) \to (e_R, e_\Omega)$ on $SO(3)$

For $R^\epsilon = R_d \exp(\epsilon \eta^\wedge)$ and $\Omega^\epsilon = \Omega_d + \epsilon \delta\Omega$ with $\delta\Omega = \Omega_d^\wedge \eta + \dot{\eta}$, we verify:
- $\|e_R^\epsilon - \epsilon \eta\| = \mathcal{O}(\epsilon^2)$
- $\|e_\Omega^\epsilon - \epsilon(\delta\Omega - \Omega_d^\wedge \eta)\| = \|e_\Omega^\epsilon - \epsilon\dot{\eta}\| = \mathcal{O}(\epsilon^2)$.

In [ ]:
Rd = random_so3(rng)
Omd = np.array([0.8, -0.4, 1.1])
eta = np.array([0.5, -0.7, 0.3])
eta_dot = np.array([-0.2, 0.6, 0.4])
dOm = np.cross(Omd, eta) + eta_dot

epsilons = np.logspace(-6, -1, 20)
err_eta = []
err_dOm = []

for eps in epsilons:
    R_eps = Rd @ exp_so3(eps * eta)
    Om_eps = Omd + eps * dOm
    eR_eps = 0.5 * vee(Rd.T @ R_eps - R_eps.T @ Rd)
    eOm_eps = Om_eps - R_eps.T @ Rd @ Omd

    err_eta.append(np.linalg.norm(eR_eps - eps * eta))
    # Equivalently || (eOm_eps / eps + Omd x eta) - dOm || * eps
    err_dOm.append(np.linalg.norm(eOm_eps - eps * (dOm - np.cross(Omd, eta))))

err_eta = np.array(err_eta)
err_dOm = np.array(err_dOm)

slope_eta = np.polyfit(np.log10(epsilons), np.log10(err_eta), 1)[0]
slope_dOm = np.polyfit(np.log10(epsilons), np.log10(err_dOm), 1)[0]
print(f'Log-log convergence slope for e_R:     {slope_eta:.3f} (expected ~2.0, i.e., O(eps^2))')
print(f'Log-log convergence slope for e_Omega: {slope_dOm:.3f} (expected ~2.0, i.e., O(eps^2))')
assert 1.9 < slope_eta < 3.5 and 1.9 < slope_dOm < 2.2

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(epsilons, err_eta, 'o-', label=r'$\|e_R^\epsilon - \epsilon\eta\|$')
ax.loglog(epsilons, err_dOm, 's-', label=r'$\|e_\Omega^\epsilon - \epsilon(\delta\Omega - \Omega_d^\wedge \eta)\|$')
ax.loglog(epsilons, 0.5 * epsilons**2, 'k--', label=r'$\mathcal{O}(\epsilon^2)$ reference')
ax.set_xlabel(r'Perturbation scale $\epsilon$')
ax.set_ylabel('Error norm')
ax.set_title('First-Order Equivalence Convergence on SO(3)')
ax.grid(True, which='both', ls='--', alpha=0.4)
ax.legend()
plt.tight_layout()
plt.show()